# 09 — Guardrails & hooks

Keyword block, PII redact, pipeline hooks, text transforms, sentence chunker.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


### Local mode
Construct a Patter instance with a Twilio carrier.


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises guardrail construction and PipelineHooks.


### `guardrail()` factory


In [ ]:
import { guardrail } from "getpatter";
await cell('guardrail_decorator', { tier: 1, env }, () => {
  const noCompetitor = guardrail({
    name: 'no_competitor_mention',
    handler: (text: string) => {
      if (text.toLowerCase().includes('rival-co')) return 'I cannot discuss that.';
      return null;
    },
  });
  const allowed = noCompetitor.handler('Hello, how can I help?');
  const blocked = noCompetitor.handler('Have you tried rival-co?');
  console.log(`allowed: ${allowed}  blocked: ${blocked}`);
  if (allowed !== null) throw new Error('expected null for allowed text');
  if (blocked === null) throw new Error('expected non-null for blocked text');
});


### Agent with guardrails


In [ ]:
import { Patter, Twilio, OpenAIRealtime, guardrail } from "getpatter";
await cell('guardrail_in_agent', { tier: 1, env }, () => {
  const noPricing = guardrail({
    name: 'no_pricing_talk',
    handler: (text: string) =>
      text.toLowerCase().includes('price') ? 'Contact sales for pricing.' : null,
  });
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const agent = p.agent({
    systemPrompt: 'You are a support agent.',
    engine: new OpenAIRealtime({ apiKey: 'sk-test' }),
    guardrails: [noPricing],
  });
  console.log(`Agent guardrails: ${agent.guardrails?.map((g: any) => g.name)}`);
});


### PipelineHooks


In [ ]:
import { PipelineHooks } from "getpatter";
await cell('pipeline_hooks', { tier: 1, env }, () => {
  const hooks: PipelineHooks = {
    onTranscript: async (text: string, isFinal: boolean) => { /* log */ },
    onLlmResponse: async (text: string) => text,
  };
  console.log(`PipelineHooks: onTranscript=${typeof hooks.onTranscript}`);
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Places a call with an active guardrail so a blocked phrase triggers a redirect. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
await cell('live_preflight', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env }, () => {
  console.log(`  carrier:  Twilio ${env.twilioNumber}  →  ${env.targetNumber}`);
  console.log('  guardrail: blocks any mention of "competitor"');
  console.log(`  webhook:  ${env.publicWebhookUrl || '(ngrok auto-launch)'}`);
});


### Live call with guardrail *(T4)*


In [ ]:
import { Patter, Twilio, OpenAIRealtime, guardrail } from "getpatter";
await cell('live_guardrail_call', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env }, async () => {
  const noCompetitor = guardrail({
    name: 'no_competitor',
    handler: (text: string) =>
      text.toLowerCase().includes('competitor') ? 'I cannot discuss other companies.' : null,
  });
  const p = new Patter({
    carrier: new Twilio({ accountSid: env.twilioSid, authToken: env.twilioToken }),
    phoneNumber: env.twilioNumber,
    webhookUrl: env.publicWebhookUrl,
  });
  const agent = p.agent({
    systemPrompt: 'Demo agent.',
    engine: new OpenAIRealtime({ apiKey: env.openaiKey }),
    guardrails: [noCompetitor],
  });
  try {
    await p.call(env.targetNumber, { agent, firstMessage: 'Hello! Mention a competitor to test the guardrail.', ringTimeout: env.maxCallSeconds });
    console.log('✓ Guardrail call completed');
  } finally {
    await hangupLeftoverCalls(env);
  }
});
